Import libraries

In [ ]:
import numpy as np
import time, json, math, random, copy, glob, os
from copy import deepcopy
from json import JSONEncoder
from datetime import datetime

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.ticker import MultipleLocator
import io
from PIL import Image

import pygad

import shapely
from shapely import Polygon
from shapely.geometry import Polygon, LineString
from shapely.ops import split


Helper functions

In [ ]:

class Helper():
    
    def __init__(self, config={}):
        # Canvas
        canvas_dim = config.get("canvas")
        self.canvas_w = canvas_dim[0]
        self.canvas_h = canvas_dim[1]
        self.pol_canvas = Polygon([
            (0,0), (self.canvas_w,0), (self.canvas_w,self.canvas_h), (0,self.canvas_h), (0,0)
        ])
        self.canvas_area = self.pol_canvas.area

        # Stock inventory
        inv_config = config.get("inventory")
        self.inv_stocks = {}
        for id, stock_pts in inv_config.items():
            self.inv_stocks[id] = Polygon(stock_pts)
        self.id_max = int(id)
        self.inv_count_max = config.get("max_number_of_stocks")

        # GA
        self.gene_space = range(0, 9)    # 9 actions
        self.weights = np.array(config.get("weight"))
        
        self.max_steps = config.get("max_step")
        self.num_genes = deepcopy(self.max_steps)

    def reset(self):

        # Canvas
        pol_canvas_diff = deepcopy(self.pol_canvas)    # Remaining canvas after placing stocks
        pol_union = Polygon()

        # Inventory
        pol_inv = deepcopy(self.inv_stocks)
        inv_assigned = []
        seq_stocks = []
        len_plaster = 0

        # Start!
        reward_stack = np.zeros(self.max_steps)
        self.best_return = -100
        self.output_return = 0
        seq_actions = []

        return self.num_genes, self.weights, pol_canvas_diff, pol_union, \
            self.id_max, self.inv_count_max, pol_inv, len_plaster, inv_assigned, \
                reward_stack, seq_actions, seq_stocks
    
    def _get_rewards(self, pol_stock_cur, pol_union, id_active, len_plaster):
        
        pol_union = shapely.unary_union([pol_union, pol_stock_cur])   # pol_union.area = state

        if not pol_union.is_empty:  # Skip when Step == 0 (.reset)

            # reuse: (+) reused area
            r_reuse = round(30*pol_union.area/self.canvas_area,2)

            # work efficiency: (-) longer plastering (+) meeting canvas boundary
            r_work = self.pol_canvas.boundary.intersection(pol_union.boundary).length \
                - len_plaster - self.inv_stocks[id_active].length

            if pol_union.geom_type == 'MultiPolygon':    # Non-overlapping polygons exist
        
                for polygon in pol_union.geoms:
                    for pol in polygon.interiors:     # (-) intricate points
                        r_work -= 2*len(pol.coords)
                    r_work -= 2*len(polygon.exterior.coords)
            
            else:   # Single unified polygon
                for pol in pol_union.interiors:     # (-) intricate points
                    r_work -= 2*len(pol.coords)
                r_work -= 2*len(pol_union.exterior.coords)

            r_work = round(10 * r_work / (16*self.canvas_area), 2) # normalization

        else:   # Step 0 for reset
            return np.array([0, 0, 0], dtype=np.float64)

        len_plaster += self.inv_stocks[id_active].length

        return np.array([r_reuse, r_work, 0], dtype=np.float64), pol_union, len_plaster

    def viz_final_layout(self, pol_union):

        fig = plt.figure(1, figsize=(5,5), dpi=96)
        ax = fig.add_subplot(111, xlim=(0,self.canvas_w), ylim=(0,self.canvas_h), aspect="equal")

        if pol_union.geom_type == 'MultiPolygon':    # Non-overlapping polygons exist
        
            for polygon in pol_union.geoms:
                ax.fill(*polygon.exterior.xy, color="blue", alpha=0.4, zorder=1)
                if polygon.interiors:
                    for interior in polygon.interiors:
                        ax.plot(*interior.xy, color='white', zorder=1)
        
        else:   # Single unified polygon
            ax.fill(*pol_union.exterior.xy, color="blue", alpha=0.4, zorder=1)
            if pol_union.interiors:
                for interior in pol_union.interiors:
                    ax.plot(*interior.xy, color='white', zorder=1)

        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        image = Image.open(buf).convert('RGB')

        return image

class NumpyArrayEncoder(JSONEncoder):

    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return int(obj)
        return JSONEncoder.default(self, obj)

class StockQueue:

    def __init__(self, id_max=5, count_max=5):
        self.id_max1 = id_max
        self.count_max = count_max
        
    def reset(self):
        random.seed(42)             # Comment this out if stock availability is fixed
        queue = list(range(1,self.id_max1+1))
        random.shuffle(queue)       # Comment this out if stock availability is fixed
        self.queue = queue[:self.count_max]
        self.id_max = deepcopy(self.id_max1)
        return self.queue

    def get_queue(self):
        return self.queue
    
    def update_queue(self):
        self.queue.pop(0)

    def add_queue(self):
        self.id_max += 1
        self.queue.append(self.id_max)
        return str(self.id_max)

    def get_next_stock(self):
        temp = self.queue[0]
        self.queue.pop(0)
        self.queue.append(temp)
        return str(self.queue[0])
    
    def retrieve_active(self):
        return str(self.queue[0])

def collision(pol_stock_cur, pol_canvas_diff):
    if shapely.covers(pol_canvas_diff, pol_stock_cur):
        return False
    else:
        return True

def export_results(info_stack, image, key_rewards=None):

    encoded_info = json.dumps(info_stack, cls=NumpyArrayEncoder)
    file_name = f'result_{datetime.now().strftime("%Y%m%d-%H%M")}_ga'

    with open(f'./result/{file_name}.json', 'w', encoding="utf-8") as f:
        json.dump(encoded_info, f)
        # print(file_name)
    image.save(f'./result/{file_name}_layout.jpg')

    if key_rewards != None:
        print("Major Updates:", key_rewards)

    return file_name


User-defined inputs

In [ ]:
num_generations = 300
MAX_STEPS = 100

config = {
    "canvas": [10,10],
    "inventory": {
        "1": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "2": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "3": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "4": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "5": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "6": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "7": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "8": [(0,0),(4,0),(4,4),(0,4),(0,0)],
        "9": [(0,0),(3,0),(3,5),(0,5),(0,0)],
        "10": [(0,0),(2,0),(2,4),(0,4),(0,0)],
        "11": [(0,0),(1,0),(1,5),(0,5),(0,0)],
        "12": [(0,0),(1,0),(1,4),(0,4),(0,0)],
        "13": [(0,0),(1,0),(1,3),(0,3),(0,0)],
        "14": [(0,0),(1,0),(1,3),(0,3),(0,0)],
        "15": [(0,0),(2,0),(2,3),(1,3),(1,4),(0,4),(0,0)],
        "16": [(0,0),(2,0),(2,3),(1,3),(1,4),(0,4),(0,0)],
        "17": [(0,0),(3,0),(3,2),(2,2),(2,1),(0,1),(0,0)],
        "18": [(0,0),(1,0),(1,1),(0,1),(0,0)],
        "19": [(0,0),(4,0),(4,4),(0,4),(0,0)],
        "20": [(0,0),(2,0),(2,1),(0,1),(0,0)]
    },
    "weight": [2,2,1],
    "max_number_of_stocks": 15,
    "max_step": MAX_STEPS
}


Run GA

In [ ]:
# GA with non-static availability

gene_space = range(0, 9)    # Total 9 actions
parent_selection_type = "sss"
keep_parents = 1
crossover_type = "single_point"
mutation_type = "random"
num_parents_mating = 10
sol_per_pop = 32
crossover_probability = 0.85
mutation_probability = 0.2

inv_best, action_best, seq_stocks_best, best_return, image = [], [], [], -100, True
helper = Helper(config=config)

def fitness_func(ga_instance, solution, solution_idx):

    global helper, inv_best, action_best, best_return, image

    # Init_General
    num_genes, weights, pol_canvas_diff, pol_union, \
            id_max, inv_count_max, pol_inv, len_plaster, inv_assigned, \
                reward_stack, seq_actions, seq_stocks = helper.reset()

    # Init_Inventory
    queue = StockQueue(id_max=id_max, count_max=inv_count_max)
    inv_aval = queue.reset()

    # 1 Gene w/eval
    for step, action in enumerate(solution):

        action = int(action)
        seq_actions.append(action)

        id_active = queue.retrieve_active()
        pol_stock_cur = pol_inv[id_active]

        rewards = np.array([0, 0, 0], dtype=np.float64)
        
        if action == 0:     # Placement

            if not collision(pol_stock_cur, pol_canvas_diff):   ## Valid! ##
                # Reward
                rewards, pol_union, len_plaster = helper._get_rewards(pol_stock_cur, pol_union, id_active, len_plaster)
                # Operation
                pol_canvas_diff = shapely.difference(pol_canvas_diff, pol_stock_cur)  # update canvas
                inv_assigned.append(id_active) # update inventory
                queue.update_queue()

            else:   ## Invalid ##
                rewards[2] -= 0.1

        elif action == 1:   # Next stock on queue
            
            if len(inv_aval) < 1:
                rewards[2] -= 1
            else:
                id_active = queue.get_next_stock()
                pol_stock_cur = pol_inv[id_active]
                rewards[2] -= 0.1

        elif action == 2 or action == 3:   # Cut

            # Reward prep
            eval_prev = abs(pol_canvas_diff.area - pol_stock_cur.area)

            # Operation
            if action == 2:     # Cut parallel to X-axis
                temp = list(set([y for _, y in list(pol_stock_cur.exterior.coords)]))
                if len(temp) == 2 and temp[1]-temp[0] > 1:
                    y2_stock_cur = temp[1]-1
                else:
                    y2_stock_cur = temp[1]
                cutting_line = LineString([(-100,y2_stock_cur), (100,y2_stock_cur)])

            else:               # Cut parallel to Y-axis
                temp = list(set([x for x, _ in list(pol_stock_cur.exterior.coords)]))
                if len(temp) == 2 and temp[1]-temp[0] > 1:
                    x2_stock_cur = temp[1]-1
                else:
                    x2_stock_cur = temp[1]
                cutting_line = LineString([(x2_stock_cur,-100), (x2_stock_cur,100)])

            pol_stocks_added = sorted(split(pol_stock_cur, cutting_line).geoms, key=lambda x: x.area)
            pol_stock_cur = pol_stocks_added.pop()

            # Reward
            eval_cur = abs(pol_canvas_diff.area - pol_stock_cur.area)
            if  eval_cur >= eval_prev:
                rewards[2] -= 0.5
            else:
                rewards[2] -= 0.1
                
            pol_inv[id_active] = pol_stock_cur

        elif action == 4:   # Rotation

            # Reward prep
            eval_prev = abs(pol_union.bounds[2]-pol_stock_cur.bounds[0] + pol_union.bounds[3]-pol_stock_cur.bounds[1])
            if not collision(pol_stock_cur, pol_canvas_diff):
                rewards[2] -= 1

            # Operation
            temp = shapely.affinity.rotate(pol_stock_cur, 90)
            pol_stock_cur = shapely.affinity.translate(
                temp,
                -temp.bounds[0],
                -temp.bounds[1]
            )

            # Reward
            eval_cur = abs(pol_union.bounds[2]-pol_stock_cur.bounds[0] + pol_union.bounds[3]-pol_stock_cur.bounds[1])
            if collision(pol_stock_cur, pol_canvas_diff):
                if eval_cur >= eval_prev:
                    rewards[2] -= 1
            else:
                if eval_cur >= eval_prev:
                    rewards[2] -= 0.5
                else:
                    rewards[2] += 0.2

            pol_inv[id_active] = pol_stock_cur
        
        elif action <7:     # Translate X

            # Reward prep
            eval_prev = abs(pol_canvas_diff.bounds[2] - pol_stock_cur.bounds[2])
            if not collision(pol_stock_cur, pol_canvas_diff):
                rewards[2] -= 1

            # Operation
            pol_stock_cur = shapely.affinity.translate(pol_stock_cur,(-1)**(5-action),0)

            # Reward
            eval_cur = abs(pol_canvas_diff.bounds[2] - pol_stock_cur.bounds[2])
            if collision(pol_stock_cur, pol_canvas_diff):
                if eval_cur >= eval_prev:
                    rewards[2] -= 1
                else:
                    rewards[2] += 0.1
            else:
                rewards[2] -= 0.5

            # Update Env
            pol_inv[id_active] = pol_stock_cur

        else:               # Translate Y

            # Reward prep
            eval_prev = abs(pol_canvas_diff.bounds[3] - pol_stock_cur.bounds[3])
            if not collision(pol_stock_cur, pol_canvas_diff):
                rewards[2] -= 1

            # Operation
            pol_stock_cur = shapely.affinity.translate(pol_stock_cur,0,(-1)**(7-action))

            # Reward
            eval_cur = abs(pol_canvas_diff.bounds[3] - pol_stock_cur.bounds[3])
            if collision(pol_stock_cur, pol_canvas_diff):
                if eval_cur >= eval_prev:
                    rewards[2] -= 1
                else:
                    rewards[2] += 0.1
            else:
                rewards[2] -= 0.5

            # Update Env
            pol_inv[id_active] = pol_stock_cur

        reward = round(float(np.sum(rewards*weights)), 2)

        reward_stack[step] = reward
        seq_stocks.append(id_active)
    
    # Completed all steps
    output_return = round(np.sum(reward_stack)/num_genes, 4)

    if best_return < output_return:
        best_return = output_return
        inv_best = inv_assigned
        action_best = seq_actions
        seq_stocks_best = seq_stocks
        image = helper.viz_final_layout(pol_union)

    return output_return


In [ ]:

fitness_function = fitness_func

ga_instance = pygad.GA(num_generations=num_generations,
                       num_parents_mating=num_parents_mating,
                       fitness_func=fitness_function,
                       sol_per_pop=sol_per_pop,
                       num_genes=MAX_STEPS,
                       gene_space=gene_space,
                       parent_selection_type=parent_selection_type,
                       keep_parents=keep_parents,
                       crossover_type=crossover_type,
                       crossover_probability=crossover_probability,
                       mutation_type=mutation_type,
                       mutation_probability=mutation_probability)
# save_solutions=True | save_best_solutions=True

start_time = time.time()
ga_instance.run()
end_time = time.time()
elapsed_time = round(end_time - start_time, 2)

best_solution, best_solution_fitness, best_match_idx = ga_instance.best_solution()
print(f"Actions of the best solution : {best_solution}")
print(f"Fitness value of the best solution = {best_solution_fitness}")  # best_return

info_stack = {
            "Index of the best solution" : f"gen_{ga_instance.best_solution_generation}-idx_{best_match_idx}",
            "Actions of the best solution" : best_solution,
            "Fitness value of the best solution": best_solution_fitness,
            "Time elapsed": elapsed_time
}

file_name = export_results(info_stack, image)
print(file_name)
